In [48]:
import pandas as pd
import numpy as np
from pathlib import Path

In [49]:
project_folder = Path.cwd().parent

year = 2026
bbl_folder = project_folder / f"Year_End_{year}" / "BBL_Account"
output_folder = project_folder / "Datamart"


output_folder.mkdir(parents=True, exist_ok=True)

print("Current notebook folder:", Path.cwd())
print("BBL source folder:", bbl_folder)
print("Output folder:", output_folder)
print("BBL folder exists:", bbl_folder.exists())

Current notebook folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Python_Command
BBL source folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Year_End_2026\BBL_Account
Output folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart
BBL folder exists: True


In [50]:
bbl_files = sorted(bbl_folder.rglob("*.xls"))

print(f"Total CSV files found: {len(bbl_files)}")

for file in bbl_files:
    print(file.relative_to(bbl_folder))

Total CSV files found: 6
Apr\TransactionsReport_May25_2026_667570-1.xls
Feb\statment BBL-ก.พ.69.xls
Jan\BBL-ม.ค.69.xls
June\TransactionsReport_Jun05_2026_125740-1.xls
Mar\BBL-มี.ค.69.xls
May\BBL-พ.ค.69.xls


In [51]:
sample_file = bbl_files[0]

print("Sample file:", sample_file)

sample_raw = pd.read_excel(
    sample_file,
    engine="xlrd",
    skiprows=3
)

display(sample_raw.head())
print(sample_raw.columns.tolist())

Sample file: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Year_End_2026\BBL_Account\Apr\TransactionsReport_May25_2026_667570-1.xls


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Customer,บจ. ไอ แฮฟ ซีพียู,NaN,Bank,BANGKOK BANK PUBLIC COMPANY LTD.,NaN,NaN,Cash Balance,300415.5
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Account Name,I HAVE CPU CO LTD,NaN,Bank Branch,NAKHON NAYOK,NaN,NaN,Available Balance,300415.5


['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']


In [52]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

def clean_text(value):
    """Convert cell value to a clean string."""
    if pd.isna(value):
        return ""
    return str(value).replace("\n", " ").strip()


def clean_number(series):
    """Convert debit, credit and balance columns to numeric."""
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("฿", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "-": np.nan,
            "nan": np.nan,
            "None": np.nan
        }),
        errors="coerce"
    )


def extract_account_number(row):
    """
    Find the account number from a row containing 'Account Num'.
    Usually the value is in the next column.
    """
    values = [clean_text(value) for value in row]

    for index, value in enumerate(values):
        if "ACCOUNT NUM" in value.upper():

            # Case 1: account number is in the next cell
            for next_index in range(index + 1, len(values)):
                candidate = values[next_index]

                if candidate:
                    match = re.search(
                        r"[0-9Xx\-]{6,}",
                        candidate
                    )
                    if match:
                        return match.group(0).upper()

            # Case 2: account number is in the same cell
            match = re.search(
                r"ACCOUNT\s*NUM(?:BER)?\s*[:\-]?\s*([0-9Xx\-]{6,})",
                value,
                flags=re.IGNORECASE
            )

            if match:
                return match.group(1).upper()

    return None

In [53]:
def read_bbl_statement(file_path):
    # Read the entire sheet without assuming a fixed header row
    raw = pd.read_excel(
        file_path,
        engine="xlrd",
        header=None,
        dtype=object
    )

    # Remove completely empty rows and columns
    raw = raw.dropna(axis=0, how="all")
    raw = raw.dropna(axis=1, how="all")
    raw = raw.reset_index(drop=True)

    all_transactions = []

    current_account_number = None
    current_account_name = None
    current_bank_branch = None
    current_currency = None
    current_customer = None

    row_index = 0

    while row_index < len(raw):

        row = raw.iloc[row_index]
        row_values = [clean_text(value) for value in row]
        row_upper = [value.upper() for value in row_values]

        # ---------------------------------------------------------
        # Capture account-level information
        # ---------------------------------------------------------

        if any("CUSTOMER" in value for value in row_upper):
            for column_index, value in enumerate(row_upper):
                if "CUSTOMER" in value:
                    for next_index in range(column_index + 1, len(row_values)):
                        if row_values[next_index]:
                            current_customer = row_values[next_index]
                            break

        if any("ACCOUNT NAME" in value for value in row_upper):
            for column_index, value in enumerate(row_upper):
                if "ACCOUNT NAME" in value:
                    for next_index in range(column_index + 1, len(row_values)):
                        if row_values[next_index]:
                            current_account_name = row_values[next_index]
                            break

        if any("ACCOUNT NUM" in value for value in row_upper):
            detected_account = extract_account_number(row)

            if detected_account:
                current_account_number = detected_account

        if any("BANK BRANCH" in value for value in row_upper):
            for column_index, value in enumerate(row_upper):
                if "BANK BRANCH" in value:
                    for next_index in range(column_index + 1, len(row_values)):
                        if row_values[next_index]:
                            current_bank_branch = row_values[next_index]
                            break

        if any("CURRENCY" in value for value in row_upper):
            for column_index, value in enumerate(row_upper):
                if value == "CURRENCY" or value.startswith("CURRENCY"):
                    for next_index in range(column_index + 1, len(row_values)):
                        if row_values[next_index]:
                            current_currency = row_values[next_index]
                            break

        # ---------------------------------------------------------
        # Detect gray transaction header row
        # ---------------------------------------------------------

        has_transaction = any(
            value == "TRANSACTION" or value.startswith("TRANSACTION")
            for value in row_upper
        )

        has_value_date = any(
            "VALUE DATE" in value
            for value in row_upper
        )

        has_description = any(
            "DESCRIPTION" in value
            for value in row_upper
        )

        if has_transaction and has_value_date and has_description:

            header_row = row_values

            # Make blank column names identifiable
            header_row = [
                value if value else f"unnamed_{index}"
                for index, value in enumerate(header_row)
            ]

            transaction_rows = []
            data_index = row_index + 1

            while data_index < len(raw):
                data_row = raw.iloc[data_index]
                data_values = [clean_text(value) for value in data_row]
                data_upper = [value.upper() for value in data_values]

                # Stop when the next account section starts
                if any("CUSTOMER" in value for value in data_upper):
                    break

                if any("ACCOUNT NUM" in value for value in data_upper):
                    break

                # Stop when another transaction header is found
                next_transaction_header = (
                    any(
                        value == "TRANSACTION"
                        or value.startswith("TRANSACTION")
                        for value in data_upper
                    )
                    and any(
                        "VALUE DATE" in value
                        for value in data_upper
                    )
                )

                if next_transaction_header:
                    break

                # Keep rows that have a transaction date
                first_value = data_values[0] if data_values else ""

                is_transaction_row = bool(
                    re.search(
                        r"\d{1,2}/\d{1,2}/\d{4}",
                        first_value
                    )
                )

                if is_transaction_row:
                    transaction_rows.append(data_row.tolist())

                data_index += 1

            if transaction_rows:
                section_df = pd.DataFrame(
                    transaction_rows,
                    columns=header_row
                )

                section_df["account_number"] = current_account_number
                section_df["account_name"] = current_account_name
                section_df["customer"] = current_customer
                section_df["bank_branch"] = current_bank_branch
                section_df["currency"] = current_currency
                section_df["source_file"] = Path(file_path).name
                section_df["source_folder"] = Path(file_path).parent.name

                all_transactions.append(section_df)

            row_index = data_index
            continue

        row_index += 1

    if not all_transactions:
        return pd.DataFrame()

    result = pd.concat(
        all_transactions,
        ignore_index=True,
        sort=False
    )

    return result

In [54]:
sample_file = bbl_files[0]

bbl_sample = read_bbl_statement(sample_file)

print("Rows:", len(bbl_sample))
print("Accounts:", bbl_sample["account_number"].unique())

display(bbl_sample.head())

Rows: 37
Accounts: <ArrowStringArray>
['3050XXX982', '9597XXX468']
Length: 2, dtype: str


,Transaction Date And Time,Value Date,Description,Cheque No.,Debit,Credit,Ledger Balance,Channel,Branch,account_number,account_name,customer,bank_branch,currency,source_file,source_folder
0,02/04/2026 09:01:51,02/04/2026,BBL.CARD,NaN,-679.45,0,275916.09,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr
1,02/04/2026 09:01:51,02/04/2026,BBL.CARD,NaN,-679.45,0,276595.54,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr
2,02/04/2026 09:01:51,02/04/2026,BBL.CARD,NaN,-679.45,0,277274.99,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr
3,30/04/2026 15:49:05,30/04/2026,RECEIVED ON GOODS,NaN,0.00,221908,1980770.72,Cash Management,NaN,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr
4,29/04/2026 17:26:05,29/04/2026,RECEIVED ON GOODS,NaN,0.00,217698,1758862.72,Cash Management,NaN,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr


In [55]:
bbl_sample.columns = (
    bbl_sample.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(".", "", regex=False)
)

print(bbl_sample.columns.tolist())

['transaction_date_and_time', 'value_date', 'description', 'cheque_no', 'debit', 'credit', 'ledger_balance', 'channel', 'branch', 'account_number', 'account_name', 'customer', 'bank_branch', 'currency', 'source_file', 'source_folder']


In [56]:

# Find all BBL .xls files
bbl_files = sorted(bbl_folder.rglob("*.xls"))

print(f"Total XLS files found: {len(bbl_files)}")

all_bbl_data = []
failed_files = []

for file in bbl_files:
    print(f"\nReading: {file.relative_to(bbl_folder)}")

    try:
        df = read_bbl_statement(file)

        if df.empty:
            print("  No transaction rows found")
            failed_files.append({
                "file": str(file),
                "reason": "No transaction rows found"
            })
            continue

        all_bbl_data.append(df)

        print(
            f"  Rows: {len(df):,} | "
            f"Accounts: {df['account_number'].nunique(dropna=True)}"
        )

    except Exception as error:
        print(f"  Failed: {error}")

        failed_files.append({
            "file": str(file),
            "reason": str(error)
        })

Total XLS files found: 6

Reading: Apr\TransactionsReport_May25_2026_667570-1.xls
  Rows: 37 | Accounts: 2

Reading: Feb\statment BBL-ก.พ.69.xls
  Rows: 44 | Accounts: 2

Reading: Jan\BBL-ม.ค.69.xls
  Rows: 50 | Accounts: 2

Reading: June\TransactionsReport_Jun05_2026_125740-1.xls
  Rows: 37 | Accounts: 2

Reading: Mar\BBL-มี.ค.69.xls
  Rows: 38 | Accounts: 2

Reading: May\BBL-พ.ค.69.xls
  Rows: 37 | Accounts: 2


In [57]:
if not all_bbl_data:
    raise ValueError("No BBL files were successfully processed.")

bbl_df = pd.concat(
    all_bbl_data,
    ignore_index=True,
    sort=False
)

print(f"\nTotal consolidated rows: {len(bbl_df):,}")
print(
    f"Total accounts: "
    f"{bbl_df['account_number'].nunique(dropna=True)}"
)



Total consolidated rows: 243
Total accounts: 2


In [58]:
bbl_df.columns = (
    bbl_df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(".", "", regex=False)
)

In [59]:
print(bbl_df.columns.tolist())

['transaction_date_and_time', 'value_date', 'description', 'cheque_no', 'debit', 'credit', 'ledger_balance', 'channel', 'branch', 'account_number', 'account_name', 'customer', 'bank_branch', 'currency', 'source_file', 'source_folder']


In [60]:
date_columns = [
    "transaction_date_and_time",
    "value_date"
]

for column in date_columns:
    if column in bbl_df.columns:
        bbl_df[column] = pd.to_datetime(
            bbl_df[column],
            dayfirst=True,
            errors="coerce"
        )

In [61]:
amount_columns = [
    "debit",
    "credit",
    "ledger_balance"
]

for column in amount_columns:
    if column in bbl_df.columns:
        bbl_df[column] = pd.to_numeric(
            bbl_df[column]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("฿", "", regex=False)
            .str.strip()
            .replace({
                "": np.nan,
                "-": np.nan,
                "nan": np.nan,
                "None": np.nan
            }),
            errors="coerce"
        )

In [62]:
final_columns = [
    "transaction_date_and_time",
    "value_date",
    "description",
    "cheque_no",
    "debit",
    "credit",
    "ledger_balance",
    "channel",
    "branch",
    "account_number",
    "account_name",
    "customer",
    "bank_branch",
    "currency",
    "source_file",
    "source_folder"
]

# Keep only available columns, in the required order
final_columns = [
    column
    for column in final_columns
    if column in bbl_df.columns
]

bbl_df = bbl_df[final_columns].copy()

In [63]:
bbl_df.insert(0, "bank", "BBL")

In [64]:
bbl_df = (
    bbl_df
    .sort_values(
        by=[
            "account_number",
            "transaction_date_and_time"
        ],
        ascending=[
            True,
            True
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)

In [65]:
print(f"Total rows: {len(bbl_df):,}")
print(f"Total files: {bbl_df['source_file'].nunique():,}")
print(f"Total accounts: {bbl_df['account_number'].nunique():,}")

display(bbl_df.head(20))

Total rows: 243
Total files: 6
Total accounts: 2


,bank,transaction_date_and_time,value_date,description,cheque_no,debit,credit,ledger_balance,channel,branch,account_number,account_name,customer,bank_branch,currency,source_file,source_folder
0,BBL,2026-01-03 03:01:05,2026-01-03,BBL.CARD,NaN,0.00,43435.19,663072.27,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
1,BBL,2026-01-04 03:00:49,2026-01-04,BBL.CARD,NaN,0.00,110100.82,773173.09,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
2,BBL,2026-01-05 03:00:44,2026-01-05,BBL.CARD,NaN,0.00,17269.82,790442.91,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
3,BBL,2026-01-05 04:01:25,2026-01-05,BBL.CARD,NaN,0.00,301.72,790744.63,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
4,BBL,2026-01-05 09:03:23,2026-01-05,BBL.CARD,NaN,-679.45,0.00,789385.73,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
5,BBL,2026-01-05 09:03:23,2026-01-05,BBL.CARD,NaN,-679.45,0.00,790065.18,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
6,BBL,2026-01-07 03:00:32,2026-01-07,BBL.CARD,NaN,0.00,1057.14,790442.87,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
7,BBL,2026-01-08 03:00:34,2026-01-08,BBL.CARD,NaN,0.00,696.64,791139.51,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
8,BBL,2026-01-11 03:06:55,2026-01-11,BBL.CARD,NaN,0.00,11417.08,802556.59,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
9,BBL,2026-01-12 03:00:34,2026-01-12,BBL.CARD,NaN,0.00,1094.16,803650.75,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan


In [66]:
display(
    bbl_df
    .sort_values("transaction_date_and_time")
    .groupby(
        [
            "account_number",
            bbl_df["transaction_date_and_time"].dt.to_period("M")
        ],
        group_keys=False
    )
    .head(1)
    .sort_values(
        [
            "transaction_date_and_time",
            "account_number"
        ]
    )
)

,bank,transaction_date_and_time,value_date,description,cheque_no,debit,credit,ledger_balance,channel,branch,account_number,account_name,customer,bank_branch,currency,source_file,source_folder
0,BBL,2026-01-03 03:01:05,2026-01-03,BBL.CARD,NaN,0.00,43435.19,663072.27,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
42,BBL,2026-01-04 04:01:23,2026-01-04,BBL.CARD,NaN,0.00,37315.83,2674929.18,Automatic,HEAD OFFICE,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-ม.ค.69.xls,Jan
17,BBL,2026-02-01 03:07:01,2026-02-01,BBL.CARD,NaN,0.00,12656.42,1055629.22,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,statment BBL-ก.พ.69.xls,Feb
75,BBL,2026-02-02 03:54:23,2026-02-02,Vendor Payment,NaN,-2000000.00,0.00,544449.62,Cash Management,NaN,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,statment BBL-ก.พ.69.xls,Feb
29,BBL,2026-03-02 09:01:58,2026-03-02,BBL.CARD,NaN,-679.45,0.00,277954.44,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-มี.ค.69.xls,Mar
107,BBL,2026-03-02 09:01:58,2026-03-02,BBL.CARD,NaN,-679.45,0.00,1397350.17,Automatic,HEAD OFFICE,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,BBL-มี.ค.69.xls,Mar
143,BBL,2026-04-01 15:57:05,2026-04-01,RECEIVED ON GOODS,NaN,0.00,438569.00,2645184.17,Cash Management,NaN,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr
31,BBL,2026-04-02 09:01:51,2026-04-02,BBL.CARD,NaN,-679.45,0.00,275916.09,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_May25_2026_667570-1.xls,Apr
34,BBL,2026-05-01 04:01:25,2026-05-01,BBL.CARD,NaN,0.00,26537.76,302453.85,Automatic,HEAD OFFICE,3050XXX982,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_Jun05_2026_125740-1.xls,June
177,BBL,2026-05-05 09:01:49,2026-05-05,BBL.CARD,NaN,-679.45,0.00,1980091.27,Automatic,HEAD OFFICE,9597XXX468,I HAVE CPU CO LTD,บจ. ไอ แฮฟ ซีพียู,NAKHON NAYOK,THB,TransactionsReport_Jun05_2026_125740-1.xls,June


In [67]:
print(bbl_df.columns.tolist())

['bank', 'transaction_date_and_time', 'value_date', 'description', 'cheque_no', 'debit', 'credit', 'ledger_balance', 'channel', 'branch', 'account_number', 'account_name', 'customer', 'bank_branch', 'currency', 'source_file', 'source_folder']


In [68]:
# Work on a copy
bbl_df = bbl_df.copy()

# Ensure datetime format
bbl_df["transaction_date_and_time"] = pd.to_datetime(
    bbl_df["transaction_date_and_time"],
    errors="coerce"
)

bbl_df["value_date"] = pd.to_datetime(
    bbl_df["value_date"],
    errors="coerce"
)

# Create KBANK-format columns
bbl_df["transaction_date"] = bbl_df["value_date"]

bbl_df["transaction_time"] = (
    bbl_df["transaction_date_and_time"]
    .dt.strftime("%H:%M")
    .astype("string")
)

# Convert debit and credit to numeric
bbl_df["debit"] = pd.to_numeric(
    bbl_df["debit"],
    errors="coerce"
)

bbl_df["credit"] = pd.to_numeric(
    bbl_df["credit"],
    errors="coerce"
)

bbl_df["ledger_balance"] = pd.to_numeric(
    bbl_df["ledger_balance"],
    errors="coerce"
)

# KBANK withdrawal values are positive
bbl_df["withdrawal"] = (
    bbl_df["debit"]
    .abs()
    .replace(0, pd.NA)
)

# KBANK deposit values are positive
bbl_df["deposit"] = (
    bbl_df["credit"]
    .abs()
    .replace(0, pd.NA)
)

# Rename remaining fields
bbl_df = bbl_df.rename(
    columns={
        "ledger_balance": "outstanding_balance",
        "channel": "transaction_type"
    }
)

# Arrange columns using KBANK format
col = [
    "bank",
    "account_number",
    "transaction_date",
    "transaction_time",
    "withdrawal",
    "deposit",
    "outstanding_balance",
    "transaction_type",
    "description"
]

bbl_df = bbl_df[col]

display(bbl_df.head())

,bank,account_number,transaction_date,transaction_time,withdrawal,deposit,outstanding_balance,transaction_type,description
0,BBL,3050XXX982,2026-01-03,03:01,<NA>,43435.19,663072.27,Automatic,BBL.CARD
1,BBL,3050XXX982,2026-01-04,03:00,<NA>,110100.82,773173.09,Automatic,BBL.CARD
2,BBL,3050XXX982,2026-01-05,03:00,<NA>,17269.82,790442.91,Automatic,BBL.CARD
3,BBL,3050XXX982,2026-01-05,04:01,<NA>,301.72,790744.63,Automatic,BBL.CARD
4,BBL,3050XXX982,2026-01-05,09:03,679.45,<NA>,789385.73,Automatic,BBL.CARD


In [69]:
import sqlite3
from pathlib import Path
import pandas as pd

# Convert transaction date
bbl_df["transaction_date"] = pd.to_datetime(
    bbl_df["transaction_date"],
    errors="coerce"
)

export_df = bbl_df.dropna(subset=["transaction_date"]).copy()

# Current notebook is inside Python_Command
project_root = Path.cwd().parent
raw_folder = project_root / "Datamart" / "Raw"
raw_folder.mkdir(parents=True, exist_ok=True)

for year, year_df in export_df.groupby(
    export_df["transaction_date"].dt.year
):
    year = int(year)

    database_path = raw_folder / f"bbl_{year}.db"
    table_name = f"bbl_{year}"

    with sqlite3.connect(database_path) as connection:
        year_df.to_sql(
            name=table_name,
            con=connection,
            if_exists="replace",
            index=False,
            chunksize=10_000
        )

    print(f"Exported {len(year_df):,} rows")
    print(f"Database: {database_path.resolve()}")
    print(f"Table: {table_name}")

Exported 243 rows
Database: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Raw\bbl_2026.db
Table: bbl_2026
